# Routing in RAG

What if you have **multiple** knowledge bases — say one for wind energy and one for pharmaceuticals — and a single user might ask about either? You don't want to search both every time. **Routing** is the answer: an extra step at the start that decides *where* the question should go.

We'll build a tiny LangGraph workflow:

```
              ┌→ vestas_retrieve →┐
classify  ──→ ┤                   ├→ generate → END
              └→ novo_retrieve  ──┘
```

The `classify` node uses an LLM to pick a route. A **conditional edge** then sends the state to the matching retriever.

## Setup

Your `.env` should have your `AZURE_OPENAI_API_KEY` plus the LLM creds. The Azure endpoint is filled in directly on the embeddings model below.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

from langchain_openai import ChatOpenAI, AzureOpenAIEmbeddings

llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")
embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    azure_endpoint="",  # TODO: paste your Azure endpoint here
)

print("Setup complete.")

## Step 1: Build two separate vector stores

One for **Vestas** (wind energy), one for **Novo Nordisk** (pharma). Same loader and splitter for both.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Vestas — wind energy
vestas_docs = PyPDFLoader(
    "https://www.vestas.com/content/dam/vestas-com/global/en/investor/reports-and-presentations/financial/2024/fy-2024/Vestas%20Annual%20Report%202024.pdf.coredownload.inline.pdf"
).load()
vestas_store = InMemoryVectorStore(embeddings)
vestas_store.add_documents(splitter.split_documents(vestas_docs))
print(f"Vestas store: {len(vestas_docs)} pages indexed.")

# Novo Nordisk — pharma
novo_docs = PyPDFLoader(
    "https://www.novonordisk.com/content/dam/nncorp/global/en/investors/irmaterial/annual_report/2025/novo-nordisk-annual-report-2024.pdf"
).load()
novo_store = InMemoryVectorStore(embeddings)
novo_store.add_documents(splitter.split_documents(novo_docs))
print(f"Novo Nordisk store: {len(novo_docs)} pages indexed.")

## Step 2: Define the workflow state

The state carries the conversation, the chosen route, and the retrieved docs.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.documents import Document
from langgraph.graph.message import add_messages


class RoutingState(TypedDict):
    messages: Annotated[list, add_messages]
    route: str  # "vestas" or "novo"
    documents: list[Document]

## Step 3: The `classify` node

We use **structured output** to force the LLM to pick exactly one of the two routes — no parsing, no surprises.

In [ ]:
from pydantic import BaseModel, Field


class RouteChoice(BaseModel):
    """Which knowledge base to use."""
    route: Literal["vestas", "novo"] = Field(description="Which company's annual report to query")
    reasoning: str = Field(description="One sentence explaining the choice")


classifier = llm.with_structured_output(RouteChoice)


def classify(state: RoutingState) -> dict:
    query = state["messages"][-1].content
    prompt = (
        "Pick the best knowledge base for this question.\n"
        "- `vestas`: wind turbines, renewable energy, wind power\n"
        "- `novo`: pharmaceuticals, diabetes, obesity, GLP-1, insulin\n\n"
        f"Question: {query}"
    )
    choice = classifier.invoke(prompt)
    print(f"→ Routed to {choice.route.upper()} ({choice.reasoning})")
    return {"route": choice.route}

## Step 4: The retrieval and generation nodes

One retriever per knowledge base, plus a single shared `generate`.

In [ ]:
def vestas_retrieve(state: RoutingState) -> dict:
    docs = vestas_store.similarity_search(state["messages"][-1].content, k=4)
    return {"documents": docs}


def novo_retrieve(state: RoutingState) -> dict:
    docs = novo_store.similarity_search(state["messages"][-1].content, k=4)
    return {"documents": docs}


def generate(state: RoutingState) -> dict:
    question = state["messages"][-1].content
    context = "\n\n".join(doc.page_content for doc in state["documents"])
    source = "Vestas" if state["route"] == "vestas" else "Novo Nordisk"

    response = llm.invoke([
        {"role": "system", "content": f"Answer using this context from {source}'s annual report:\n{context}"},
        {"role": "user", "content": question},
    ])
    return {"messages": [response]}

## Step 5: Wire it up

The key piece is `add_conditional_edges` — instead of always going to the same next node, it picks based on the value of `state["route"]`.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(RoutingState)
builder.add_node("classify", classify)
builder.add_node("vestas_retrieve", vestas_retrieve)
builder.add_node("novo_retrieve", novo_retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    lambda state: state["route"],  # this value picks the next node
    {
        "vestas": "vestas_retrieve",
        "novo": "novo_retrieve",
    },
)
builder.add_edge("vestas_retrieve", "generate")
builder.add_edge("novo_retrieve", "generate")
builder.add_edge("generate", END)

router = builder.compile()
router

## Step 6: Test it

Notice neither query mentions a company by name — the classifier picks the route from topic alone.

In [ ]:
result = router.invoke({
    "messages": [{"role": "user", "content": "How many offshore wind turbines were installed last year?"}],
    "route": "",
    "documents": [],
})

for message in result["messages"]:
    message.pretty_print()

In [ ]:
result = router.invoke({
    "messages": [{"role": "user", "content": "What treatments are available for diabetes?"}],
    "route": "",
    "documents": [],
})

for message in result["messages"]:
    message.pretty_print()

## Try it yourself 🛠️

Add a **third route** for general/off-topic questions — for example *"What's the weather like?"* — that doesn't hit any vector store and just lets the LLM answer directly.

1. Add `"general"` to the `Literal` in `RouteChoice` and update the classifier prompt to mention it.
2. Add a `general_answer(state)` node that calls `llm.invoke(...)` with the user's question (no retrieved context) and returns `{"messages": [response]}`.
3. Add `"general": "general_answer"` to the `add_conditional_edges` mapping, and an edge `general_answer → END`.
4. Test with: *"What's a fun fact about octopuses?"* — the router should pick `general` and skip retrieval.

**Hint:** `general_answer` can go straight to `END` — it doesn't need to flow through `generate`.

In [ ]:
# Your solution here

class RouteChoice(BaseModel):
    """Which knowledge base to use."""
    # TODO: extend the Literal with "general"
    route: Literal["vestas", "novo"] = Field(description="Which knowledge source")
    reasoning: str = Field(description="One sentence explaining the choice")


classifier = llm.with_structured_output(RouteChoice)


def classify(state: RoutingState) -> dict:
    query = state["messages"][-1].content
    prompt = (
        "Pick the best route for this question.\n"
        "- `vestas`: wind turbines, renewable energy, wind power\n"
        "- `novo`: pharmaceuticals, diabetes, obesity, GLP-1, insulin\n"
        # TODO: add a description for the `general` route
        f"\nQuestion: {query}"
    )
    choice = classifier.invoke(prompt)
    print(f"→ Routed to {choice.route.upper()} ({choice.reasoning})")
    return {"route": choice.route}


def general_answer(state: RoutingState) -> dict:
    # TODO: call llm.invoke(...) with the user's question and return {"messages": [response]}
    ...


builder = StateGraph(RoutingState)
builder.add_node("classify", classify)
builder.add_node("vestas_retrieve", vestas_retrieve)
builder.add_node("novo_retrieve", novo_retrieve)
builder.add_node("generate", generate)
# TODO: add the general_answer node

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    lambda state: state["route"],
    {
        "vestas": "vestas_retrieve",
        "novo": "novo_retrieve",
        # TODO: map "general" to "general_answer"
    },
)
builder.add_edge("vestas_retrieve", "generate")
builder.add_edge("novo_retrieve", "generate")
builder.add_edge("generate", END)
# TODO: add an edge from general_answer to END

router = builder.compile()

result = router.invoke({
    "messages": [{"role": "user", "content": "What's a fun fact about octopuses?"}],
    "route": "",
    "documents": [],
})

for message in result["messages"]:
    message.pretty_print()